# 🐱🐶 Lesson 16 — A Simple Modular CNN Project

**One notebook to understand the whole project, step by step.** The code is intentionally kept close to our lesson 015/016 tutorial. Small `.py` modules hold the reusable parts; this notebook lets you understand and execute them interactively in **VS Code**.

### What will we do?

`Images → Custom Dataset → DataLoader → CNN → Training → Test Accuracy → Save .pth → Predict`

> **Remember:** Dataset prepares **one** sample; DataLoader provides a **batch**; CNN learns image features; `ImageClassifier` uses trained weights on a new image.

## 0. VS Code setup — find our project folder

The code below makes imports work whether VS Code starts Jupyter in `Computer_Vision`, `CV_Modular`, or the `notebook` directory. **Only run this setup cell once.** Nothing changes your original notebooks.

In [ ]:
from pathlib import Path
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Find the modular folder even if Jupyter starts in a different directory.
folder = Path.cwd()
if (folder / "CV_Modular" / "src").is_dir():
    folder = folder / "CV_Modular"
elif (folder / "src").is_dir():
    pass
elif (folder.parent / "src").is_dir():
    folder = folder.parent
else:
    raise FileNotFoundError("Open this project in VS Code from Computer_Vision/CV_Modular.")
sys.path.insert(0, str(folder))
print("Our project:", folder)

## 1. Dataset — **how do we read ONE image?**

Open [`src/components/data_ingestion.py`](../src/components/data_ingestion.py) and read the three methods:

| Method | Easy memory tip |
|---|---|
| `__init__` | **Collect** image paths and corresponding labels. |
| `__len__` | **Count** images. |
| `__getitem__` | **Load one** image, apply Transform, and return its label. |

The exact three folders in this course dataset are `Cat/`, `Dog/`, `person/`. `sorted()` assigns consistent labels for both Train and Test.

In [ ]:
from src.components.data_ingestion import ImageDataset
from src.components.data_transformation import transform

# Dataset remains in your EXISTING PyTorch folder, next to CV_Modular.
dataset_dir = folder.parent / "PyTorch" / "Classification_dataset_v3"
train_image_dir = dataset_dir / "images" / "train"
test_image_dir = dataset_dir / "images" / "test"

train_image_dataset = ImageDataset(image_dir=str(train_image_dir), transform=transform)
test_image_dataset = ImageDataset(image_dir=str(test_image_dir), transform=transform)

print("Classes:", train_image_dataset.class_name)
print("Training images:", len(train_image_dataset))
print("Testing images:", len(test_image_dataset))

## 2. Transform — **make all images ready for the CNN**

Open [`src/components/data_transformation.py`](../src/components/data_transformation.py). We preserve the lesson's three steps:

- `Resize((128,128))` → make every image 128 × 128.
- `ToTensor()` → convert to a Tensor shaped `(3,128,128)` (RGB channels first).
- `Normalize(mean=0.5, std=0.5)` → normalize pixel values (we must use the **same** steps for new images later).

In [ ]:
# Get ONE image without DataLoader.
image, label = train_image_dataset[0]
print("One image:", image.shape)
print("True class:", train_image_dataset.class_name[label])

# Undo Normalize only for VISUALIZATION (not for CNN training).
img = np.transpose(image.numpy(), (1, 2, 0))
img = np.clip(img * 0.5 + 0.5, 0, 1)
plt.imshow(img)
plt.title(train_image_dataset.class_name[label])
plt.axis("off")
plt.show()

## 3. DataLoader — **collect samples into Batches**

`batch_size=32` means one iteration receives **32 images + 32 labels**. `shuffle=True` mixes the order for training; test order can stay fixed.

The shape `(32,3,128,128)` means **Batch, Channels, Height, Width** (BCHW). The last batch can have fewer than 32 items.

In [ ]:
# Same simple DataLoader code used in lesson 015/016.
train_image_loader = DataLoader(dataset=train_image_dataset, batch_size=32, shuffle=True)
test_image_loader = DataLoader(dataset=test_image_dataset, batch_size=32, shuffle=False)

# Inspect ONE batch, not the whole dataset.
for images, labels in train_image_loader:
    print("Images:", images.shape)
    print("Labels:", labels.shape)
    break

print("Iterations in one epoch:", len(train_image_loader))

## 4. CNN — **extract features, then predict one of 3 classes**

Open [`src/components/model.py`](../src/components/model.py): this is the **same four CNN blocks** from the lesson.

```text
Image: 3 × 128 × 128
 ↓ Conv + BatchNorm + ReLU + MaxPool
32 × 64 × 64
 ↓ second block
64 × 32 × 32
 ↓ third block
128 × 16 × 16
 ↓ fourth block
256 × 8 × 8
 ↓ Flatten (256×8×8 = 16,384 values)
Linear: 16,384 → 512 → 128 → 3 class scores
```

**What to remember:** `__init__` builds layers; `_get_conv_output` finds the flattened size using a dummy image; `forward` says how the real image moves through the network.

In [ ]:
from src.components.model import CustomCnnModel

# Use GPU when available; otherwise CPU.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CustomCnnModel(input_dim=128, num_classes=3).to(device)
print("Device:", device)
print("Flatten features:", model._to_linear)  # 16384

# Check one image -> three class scores (logits), not yet probabilities.
model.eval()
with torch.no_grad():
    scores = model(images[:1].to(device))
print("One-image output:", scores.shape)  # (1, 3)

## 5. Training — **teach the model using batches**

Open [`src/components/model_trainer.py`](../src/components/model_trainer.py). The central five lines are the original lesson's loop:

```python
optimizer.zero_grad()       # Remove previous gradients.
outputs = model(images)     # CNN predicts three class scores.
loss = criterion(outputs, labels)  # Compare scores to true labels.
loss.backward()             # Backpropagation calculates gradients.
optimizer.step()            # Adam updates the weights.
```

**Epoch vs iteration:** One **iteration** uses one batch; one **epoch** processes all batches. Your lesson uses **40 epochs**. For a quick run, start with `epochs=1`, then change to `40` for a full training run.

> ⚠️ This cell trains the model and may be slow on CPU. Only run it when you're ready.

In [ ]:
from src.components.model_trainer import train_model, evaluate_model

# Change to 40 to match the full original lesson.
epochs = 1
train_model(model, train_image_loader, device, epochs=epochs)

## 6. Evaluation — **how often are predictions correct?**

Testing is **not** training. `model.eval()` selects evaluation behavior (important for BatchNorm); `torch.no_grad()` avoids computing gradients. `torch.max(outputs,1)` chooses the highest-scoring of the three classes.

`accuracy = 100 × correct / total`

In [ ]:
accuracy = evaluate_model(model, test_image_loader, device)

## 7. Save — **keep the learned CNN weights**

`.pth` is a common PyTorch filename extension. `model.state_dict()` contains the trained model weights and buffers, **not** the class code. We need the same `CustomCnnModel` architecture again when loading.

In [ ]:
# Save trained weights for future predictions (after training).
model_path = folder / "artifacts" / "cnn_model.pth"
model_path.parent.mkdir(exist_ok=True)
torch.save(model.state_dict(), str(model_path))
print("Saved:", model_path)

## 8. Prediction — **reuse our trained model on a new image**

Open [`src/pipeline/predict_pipeline.py`](../src/pipeline/predict_pipeline.py). The `ImageClassifier` is **not a second CNN**: it *creates the same CNN*, *loads saved weights*, and *predicts one new image*.

`unsqueeze(0)` adds the batch dimension: `(3,128,128) → (1,3,128,128)`. `cv2.putText` only writes the predicted label onto a copy of the image; **OpenCV does not make the prediction**.

> Training and prediction share the **same `transform`**. Otherwise the CNN receives images with different value scales.

In [ ]:
from src.pipeline.predict_pipeline import ImageClassifier

# Choose a known test picture only to CHECK the model's prediction.
image_path = dataset_dir / "images" / "test" / "Dog" / "dog_1000.jpg"
classifier = ImageClassifier(str(model_path), train_image_dataset.class_name)
output_path = folder / "artifacts" / "output_image.jpg"
label = classifier.predict(str(image_path), str(output_path))
print("Predicted class is:", label)
print("Annotated image saved to:", output_path)

## ✅ One-page summary — always remember

| Task | File | Remember |
|---|---|---|
| Collect images + labels | `data_ingestion.py` | Dataset prepares **ONE** sample. |
| Resize + Tensor + Normalize | `data_transformation.py` | Same transform for train and inference. |
| Create batches | `DataLoader` in notebook/train pipeline | Batch of 32 means one iteration processes 32 images. |
| Build CNN | `model.py` | Conv → BN → ReLU → Pool × 4 → Flatten → Linear. |
| Learn from images | `model_trainer.py` | Forward → Loss → Backward → Optimizer. |
| Evaluate | `model_trainer.py` | Compare predicted vs true labels on **test data**. |
| Save weights | `.pth` | Avoid retraining every time you predict. |
| Predict new images | `predict_pipeline.py` | Load CNN + weights → predict → `cv2` writes label. |

**Industrial connection:** The same structure can later use images of materials, surfaces or manufactured components. The actual accuracy must be validated with real industrial images; this notebook is a **learning project**, not an industry-certified model.